# TEAM08 전체 캘리브레이션 — 반드시 위에서부터 한 셀씩 실행

이 노트북은 정상 주행용 `../dpu/dpu.bit`과 `0x00A0...` 모터 주소를 먼저 로드한 뒤 측정합니다.

- 조향 ADC 실제 측정 3점: 최대 우 / 정확한 직진 중앙 / 최대 좌
- 우·좌 HALF는 눈대중 측정 없이 세 기준점에서 자동 계산
- PWM **100% 고정** 조향 방향 검증
- 3개 자동 시작 위치 × 좌우 방향 × 6개 펄스 길이 × 3회 반복
- PWM 종료 후 관성 이동량 측정과 목표 ADC 도달 검증
- 2차선 IN/OUT 접촉·안전 한계와 이상적 직선 위치

**Run All 금지.** 조향 시험 동안 앞바퀴는 반드시 띄웁니다.


## 0. 준비

1. 차량의 뒷바퀴가 바닥을 밀지 않도록 고정합니다.
2. 조향 캘리브레이션 동안 **앞바퀴를 바닥에서 완전히 띄웁니다.**
3. 카메라와 가변저항, 모터 배선은 실제 경기 상태 그대로 둡니다.
4. 이 폴더에 예전 `calibration_result.json`이 있어도 새 결과로 덮어쓰지만, 실행 중에는 커널을 재시작하지 않습니다.
5. 아래 셀 실행 직후 모든 모터는 OFF 상태입니다.

In [ ]:
from calibration_helpers import CalibrationSession
session = CalibrationSession()
session.check_paths()

## 1. ADC 실제 기준점 3개 측정

- **Right FULL**: 기계적으로 가능한 최대 우측
- **CENTER**: 긴 자나 곧은 각재를 앞바퀴 옆면에 대고 차체 중심선과 정확히 평행하게 만든 위치
- **Left FULL**: 기계적으로 가능한 최대 좌측

Right/Left HALF는 사람이 맞추지 않습니다. 코드가 `FULL↔CENTER` ADC 중간값으로 자동 생성하고 펄스로 해당 위치에 이동합니다.


In [ ]:
session.adc_capture_widget()

세 버튼을 완료한 뒤 실행합니다. 출력은 반드시
`right_full < center < left_full`이어야 합니다. `right_half`, `left_half`에는 `derived: true`가 표시됩니다.


In [ ]:
adc_report = session.adc_report()
session.apply_adc_to_session()

## 2. 좌/우 모터 방향 검증

앞바퀴를 계속 띄운 상태에서 실행합니다. 중앙으로 자동 복귀한 뒤 12 ms짜리 PWM 100% 펄스를 좌·우 각각 한 번 줍니다.

정상 기준:
- `left` 펄스 → ADC 증가
- `right` 펄스 → ADC 감소

FAIL이면 이후 셀을 절대 진행하지 않습니다.

In [ ]:
direction_result = session.verify_steering_directions(pulse_ms=12)

## 3. 전체 PWM 100% 펄스·관성 매트릭스

앞바퀴를 계속 띄웁니다. 이 셀은 자동으로 다음 **108회**를 수행합니다.

- 시작 위치: Right HALF / CENTER / Left HALF
- 방향: LEFT / RIGHT
- 펄스: 8, 12, 16, 24, 32, 48 ms
- 반복: 각 3회
- 기록: 펄스 전, 종료 직후, 20/50/100/200 ms 후 ADC

각 시험 전에는 짧은 100% 펄스로 시작 위치에 자동 복귀합니다. 뒷바퀴 모터는 켜지지 않습니다. 약 수 분 걸립니다.

In [ ]:
pulse_model = session.run_full_pulse_matrix()

## 4. 목표 ADC 도달 검증

이제 방금 만든 펄스 모델을 실제 제어기처럼 사용합니다.

`0 → +1.75 → +3.5 → +5 → 0 → -1.75 → -3.5 → -5 → 0`을 3회 반복하여, 직선부터 최대 주행 조향 범위까지 목표 위치에 도달하는지 확인합니다.

이 단계가 PASS하지 않으면 최종 파일은 생성되지 않습니다.

In [ ]:
position_validation = session.validate_target_positioning(repeats=3)

## 5. 차선 접촉·안전 한계 5점 측정

이제 앞바퀴를 내려놓고 차량 전원을 안전하게 유지하되, 차량은 **손으로만 이동**합니다. 직선 2차선에서 차체와 차선을 완전히 평행하게 둡니다.

측정 순서:

1. **IN TOUCH**: 안쪽 타이어가 중앙 점선에 막 닿기 시작하는 위치
2. **IN SAFE**: IN TOUCH에서 차로 안쪽으로 정확히 약 30 mm 이동
3. **IDEAL**: 평소 직선에서 달리고 싶은 위치
4. **OUT SAFE**: OUT TOUCH에서 차로 안쪽으로 약 30 mm 이동한 위치
5. **OUT TOUCH**: 바깥 타이어가 바깥 실선에 막 닿기 시작하는 위치

각 위치에서 차량의 yaw가 조금이라도 바뀌면 측정을 다시 합니다. 카메라 마운트도 절대 움직이지 않습니다. 규정상 바깥 2차선은 중앙 점선이 IN, 바깥 실선이 OUT입니다.

In [ ]:
session.lane_capture_widget()

다섯 위치를 모두 캡처한 뒤 실행합니다. TOUCH 범위가 SAFE 범위를 감싸고 IDEAL이 SAFE 안에 있어야 합니다.

In [ ]:
lane_report = session.lane_report()

## 6. 선택 사항 — 차량 치수와 속도

이 값은 주행 코드의 핵심 캘리브레이션에는 필요하지 않습니다. 시뮬레이션을 실제 차량에 더 가깝게 맞출 때만 입력합니다.

예: duty 100으로 직선 2 m를 주행한 시간이 2.3초라면 `measured_distance_m=2.0`, `measured_time_s=2.3`을 입력합니다.

In [ ]:
# 필요할 때만 숫자를 바꾸고 실행하세요. 생략해도 됩니다.
# session.set_vehicle_measurements(
#     wheelbase_m=0.64,
#     track_width_m=0.50,
#     tire_width_m=0.09,
#     measured_distance_m=2.0,
#     measured_time_s=2.3,
# )

## 7. 최종 검사 및 파일 생성

모든 항목이 `true`인지 먼저 확인합니다. 하나라도 `false`면 해당 단계를 다시 수행합니다.

In [ ]:
session.pre_export_check()

아래 셀은 다음을 생성합니다.

- `calibration_result.json`: 실제 주행 코드가 자동으로 읽는 파일
- `calibration_output/TEAM08_CALIBRATION_UPLOAD.zip`: 대화에 다시 올릴 전체 결과
- ADC/펄스/목표 도달/차선 CSV와 차선 캡처 이미지

생성 후 **커널을 반드시 재시작**해야 `config.py`가 새 결과를 읽습니다.

In [ ]:
upload_zip = session.export()
print('UPLOAD THIS FILE:', upload_zip)

## 8. 종료

파일 생성이 끝났으면 리소스를 닫습니다. 그다음 Kernel → Restart를 누르고 `RUN_TEAM08.ipynb`를 실행합니다.

In [ ]:
session.close()